# CLI Functions

> Functions for use in the vnn.py cli

In [ ]:
#| default_exp clifn

In [ ]:
#| export
import pandas as pd
import pyarrow.parquet as pq

import argparse, os, re, sparsevnn
import torch
import torch.nn.functional as F
import sparsevnn
import sparsevnn.util
import sparsevnn.qol
from   sparsevnn.core import \
    info_list_to_layer_list, \
    SparseVNN,               \
    structured_layer_info

from   sparsevnn.core import plDNN_general

import lightning.pytorch as pl
from   lightning.pytorch.loggers import CSVLogger # used to save the history of each trial (used by ax)


# used in eval functions
import numpy  as np
import torch.nn as nn
import plotly.express as px
import scipy.stats # for spearmanr
import tqdm


In [ ]:
#| hide
from nbdev.showdoc import *

## For argparse

In [ ]:
#| export
def s2b(v):
    # Note type bool does not interpret the text following the flag as a bool, rather _passing_ the flag is a bool test
    # See https://stackoverflow.com/questions/60999816/argparse-not-parsing-boolean-arguments
    # and https://stackoverflow.com/questions/15008758/parsing-boolean-values-with-argparse/43357954#43357954
    "Credit to user [Maxim](https://stackoverflow.com/users/805502/maxim) "
    if isinstance(v, bool):
        return v
    if v.lower() in ('yes', 'true', 't', 'y', '1'):
        return True
    elif v.lower() in ('no', 'false', 'f', 'n', '0'):
        return False
    else:
        raise argparse.ArgumentTypeError('Boolean value expected.')


In [ ]:
#| export
def as_lat(inp):
    # If  --eval a b        -> [[a, b]]
    #     --eval a --eval b -> [[a], [b]]
    # so as_lat ensures that instead we have a flat list as `eval`
    def lat(x):
        True if list not in [type(e) for e in x] else False

    # This used to be a while but if inp is a [[]] list it starts an infinite loop. 
    # This behavior isn't one I observed until recently. (250128)
    for i in range(1000): #arbitrarily high limit 
        if lat(inp):
            break
        else:
            # turn everything that isn't a list into one, then use sum to concat the lists
            inp = sum([[e] if type(e) is not list else e for e in inp], [])
            # the case that was causing the infinite loop was a length 1 list containing a list
            if ((len(inp) == 1) & (type(inp[0]) is list)):
                inp = inp[0]
    return inp

In [ ]:

#| export
def _get_json_if_exists(
        path, 
        file, # File name or regex (e.g. '\.gff$')
        is_regex = False
        ):
    res = {}
    files = os.listdir(path=path)
    if is_regex:
        file = sorted([e for e in files if re.match(file, e)])
        if file != []:
            file = file[0] 
            res = sparsevnn.qol.read_json(json_path=path+file)
    elif file in files:
            res = sparsevnn.qol.read_json(json_path=path+file)

    if res != {}: 
        print(f'Loading and using {path+file}.')
    return res

## Customizable Training Functions

In [ ]:
#| export

def train_one_model(
    params, # = params,
    params_data, # = params_data,
    params_run, # = params_run,
    edge_dict, # = cxn_dict,
    inp_tensor_lookup, # = inp_node_idx_dict,
    log_dir, # = lightning_log_dir # This is explicit istead of using the global scope so that hyps/training can log different dirs. 
    exp_name,            # NEW
    training_dataloader, # NEW
    validation_dataloader# NEW
    ):
    def _propose_model(
        edge_dict = edge_dict,
        params_data = params_data,
        inp_tensor_lookup = inp_tensor_lookup,
        params = params
        ):
        myvnn = sparsevnn.util.mk_vnnhelper(
                edge_dict = edge_dict,
                num_nucleotides = params_data['num_nucleotides'], # this could also be 1 for major/minor allele. 
                inp_tensor_lookup = inp_tensor_lookup,
                params = params
                    )

        dd = sparsevnn.core.mk_NodeGroups(edge_dict=myvnn.edge_dict, dependancy_order=myvnn.dependancy_order)

        M_list = [
            structured_layer_info(
            i = ii, 
            node_groups=dd, 
            node_props=myvnn.node_props, 
            edge_dict=myvnn.edge_dict, 
            as_sparse=True,
            inp_tensor_nucleotides= params_data['num_nucleotides'],
            # lambda to only provide the lookup for the 0th grouping (input level)
            inp_tensor_lookup = (lambda x: inp_tensor_lookup if x == 0 else None)(ii)
            )
            for ii in sorted(list(dd.keys()))]

        # layer_list = info_list_to_layer_list(M_list = M_list, nonlinearity = F.relu)
        layer_list = info_list_to_layer_list(M_list = M_list, nonlinearity = F.tanh)
        model      = SparseVNN(layer_list = layer_list)
        return M_list, model
    
    # with relu there is a chance that the model in initialled fully in a null field. 
    # Test and retry models until there's an okay initialization.
    def calc_dl_yhat_stats(dl, model, stats = ['pr_uniq']):
        out_stats = {}
        # assume that the genome tensor was deduplciated (should be because we're reading from a file) (exception: error in names)
        # check if matches have the same input?
        # I'm re-implmenting some features we get for free in the dataloader. 
        bs = dl.batch_size # batch size
        gs = dl.dataset.G.size()[0] # obs
        mb = [i for i in range(0, gs, bs)] # minibatch idxs
        if mb[-1] != [gs]: 
            mb = mb + [gs] # add final stop index (len)

        mb = [(i,j) for i,j in zip(mb, mb[1:])]

        model = model.eval()
        yhats = []
        for (i,j) in mb:
            yhat = model(dl.dataset.G[i:j])
            yhats.append(yhat.detach().cpu())
        model = model.train()

        yhats = torch.concat(yhats)

        if 'pr_uniq' in stats: # calculate percent unique
            pr_uniq = torch.unique(yhats).shape[0] / yhats.shape[0]
            out_stats['pr_uniq'] = pr_uniq

        if 'std' in stats:
            out_stats['std'] = float(yhats.std())

        return out_stats


    pr_uniq_threshold  = 0.95 # at .99 failed with F.relu and 100 iterations. Leaky relu also failed. 
    check_model_trials = 100

    tmp = []
    for check_model_i in range(check_model_trials):
        model_M_list, model = _propose_model()
        model = model.to('cuda')
        model.eval()
        # we should only really need to do this for the training dataloader because we use the same data underthe hood for both
        pr_uniq = calc_dl_yhat_stats(dl = training_dataloader, model = model, stats = ['pr_uniq'])['pr_uniq']
        tmp.append(pr_uniq)
        if pr_uniq >= pr_uniq_threshold:
            break

    print("\nyhat distribution check:")    
    print("\npass\ttrials\tmax\tthreshold")
    print(f"{pr_uniq >= pr_uniq_threshold}\t{len(tmp)}\t{check_model_trials}\t{pr_uniq_threshold}")
    print("\n") 
    print("pr_uniq recorded:")   
    print(f"{tmp}")
    print("\n")    

    del tmp
    if not (pr_uniq >= pr_uniq_threshold):
        print('Initialization condition not met. Consider modifying `pr_uniq_threshold`. Breaking.')
        # NOTE: this will break (intentionally). 
        # assert(True == False)
        # None[0] 
        # per conversation with Jacob (25/01/16) if we don't meet the criteria in 100 tries we'll allow the model to run anyway. 
        # The expectation is that the fit will be poor (in the worst case we'll fit an intercept model) resulting in 
        # the hyperparameter set _underperforming_ the likely "true" performance (conditional on correct initialization).
        # This in effect means the tuner is penalizing 1) poor performance and 2) difficulty of initialization. 
        print('Proceeding anyway.') 

    if True:
        VNN        = plDNN_general(model)
        VNN.configure_optimizers()
        # optimizer = VNN.configure_optimizers()
        logger    = CSVLogger(log_dir, name=exp_name)
        logger.log_hyperparams(params={
            'params': params,
            'params_data': params_data,
            'params_run': params_run
        })
        trainer = pl.Trainer(max_epochs=params_run['max_epoch'], logger=logger)
        trainer.fit(model=VNN, train_dataloaders=training_dataloader, val_dataloaders=validation_dataloader)
        return model_M_list, trainer # NOTE -----------------------------------------------
        # return trainer

In [ ]:
#| export

def vnn_from_state_dict(
    params, # = params,
    params_data, # = params_data,
    edge_dict, # = cxn_dict,
    inp_tensor_lookup, # = inp_node_idx_dict,
    state_dict_path = None,
    ):
    myvnn = sparsevnn.util.mk_vnnhelper(
            edge_dict = edge_dict,
            num_nucleotides = params_data['num_nucleotides'], # this could also be 1 for major/minor allele. 
            inp_tensor_lookup = inp_tensor_lookup,
            params = params
                )

    dd = sparsevnn.core.mk_NodeGroups(edge_dict=myvnn.edge_dict, dependancy_order=myvnn.dependancy_order)

    M_list = [
        structured_layer_info(
        i = ii, 
        node_groups=dd, 
        node_props=myvnn.node_props, 
        edge_dict=myvnn.edge_dict, 
        as_sparse=True,
        inp_tensor_nucleotides= params_data['num_nucleotides'],
        # lambda to only provide the lookup for the 0th grouping (input level)
        inp_tensor_lookup = (lambda x: inp_tensor_lookup if x == 0 else None)(ii)
        )
        for ii in sorted(list(dd.keys()))]

    layer_list = info_list_to_layer_list(M_list = M_list, nonlinearity = F.tanh)
    model      = SparseVNN(layer_list = layer_list)
    # now we load the state dict into the model
    model.load_state_dict(torch.load(state_dict_path))
    return model

In [ ]:
#| export

def evaluate(
        parameterization, 
        cxn_dict,
        inp_node_idx_dict,
        lightning_log_dir,
        exp_name, 
        params_data,
        params_run,
        training_dataloader,
        validation_dataloader,

        ):
    "This is for Ax's use which is why it pulls variables from the global scope."
    _ = train_one_model(
        params = parameterization,
        edge_dict = cxn_dict,
        inp_tensor_lookup = inp_node_idx_dict,
        log_dir = lightning_log_dir,
        exp_name = exp_name,
        params_data = params_data,
        params_run = params_run,
        training_dataloader = training_dataloader,
        validation_dataloader = validation_dataloader
    )
    # if we were optimizing number of training epochs this would be an effective loss to use.
    # trainer.callback_metrics['train_loss']
    # float(trainer.callback_metrics['train_loss'])
    # To potentially _overtrain_ models and still let the selction be based on their best possible performance,
    # I'll use the lowest average error in an epoch
    log_path = lightning_log_dir+'/'+exp_name
    fls = os.listdir(log_path)
    nums = [int(e.split('_')[-1]) for e in fls] 

    M = pd.read_csv(log_path+f"/version_{max(nums)}/metrics.csv")
    M = M.loc[:, ['epoch', 'train_loss']].dropna()

    M = M.groupby('epoch').agg(
        train_loss = ('train_loss', 'mean'),
        train_loss_sd = ('train_loss', 'std'),
        ).reset_index()

    train_metric = M.train_loss.min()
    print(train_metric)
    _ = {"train_loss": (train_metric, 0.0)}
    return _



In [ ]:
#| export


# Turn all the tensors holding shape and index info into ints. This _massively_ reduces the space to save it.
def _shrink_M_list(M_list): 
    # storing col and row info as tensors is convenient but has a massive effect on storage size
    # If all the size/start/stop values are coerced to ints the size for a test M_list goes from
    # 5.2G -> 7.5M. At that size we might as well save out al the attributes in it.
    for i in range(len(M_list)):
        e = M_list[i].col_info
        M_list[i].col_info = {k: {
            'size':int(e[k]['size']),
            'start':int(e[k]['start']),
            'stop':int(e[k]['stop']),
            } 
            for k in e.keys()
        }
        
        e = M_list[i].row_info
        M_list[i].row_info = {k: {
            'size':int(e[k]['size']),
            'start':int(e[k]['start']),
            'stop':int(e[k]['stop']),
            } 
            for k in e.keys()
        }
    return M_list



## Functions for making predictions

In [ ]:
#| export


def _collect_predictions(model, inp_dl):
    # check that model is on the same device as the data
    if next(iter(inp_dl))[0].get_device() != -1:
        model = model.to('cuda')

    # TODO test with 
    model.eval()    
    # as a sanity check we'll save the true y's. That will allow for 
    yvar, yhat = [], []
    # only get the validation dataloader if the training dataloader is shuffled.
    for i, (y,x) in enumerate(inp_dl):
        yvar.append(       y.detach().cpu() )
        yhat.append(model(x).detach().cpu() )

    yvar = torch.concat(yvar)
    yhat = torch.concat(yhat)
    return yvar, yhat


## Functions for model evaluation

In [ ]:
#| export

### Input Saliences ####
# iterate over dataloader and aggregate all of the saliences for the input data
def _collect_salience_snpwise(model, inp_dl, params_data):
    # check that model is on the same device as the data
    if next(iter(inp_dl))[0].get_device() != -1:
        model = model.to('cuda')

    # get saliency for all obs given model, dataloader
    def _get_saliency(y_i, x_i, model):
        x_i.requires_grad_()
        model.eval()

        optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
        loss_fn = nn.MSELoss()

        loss = loss_fn(model(x_i), y_i)
        optimizer.zero_grad()
        loss.backward()
        out = x_i.grad
        out = out.to('cpu').numpy()
        # for output size 
        # reshape to b,n,l
        out = out.reshape(out.shape[0], -1, params_data['num_nucleotides']).swapaxes(1,2)
        # reduce to max over nucleotide axis
        out = out.max(axis = 1) # max over nucleotide axis
        out.shape
        return out

    out = []
    for i, (y,x) in enumerate(inp_dl):
        out.append(_get_saliency(y_i = y, x_i = x, model = model))
    out = np.concatenate(out)

    #reverse workup for acgt_tensor to get (obs, nuc, len)
    return out



In [ ]:
#| export


### SNP-wise Manhattan 
def _plt_saliences(salience, save_dir,  plt_prefix):
    print('\n'.join(['Percentiles:']+[f'q {i} = {np.quantile(salience.salience, q = i)}' for i in [.95, .99, .999]]))
    
    # salience distribution
    plt_d = px.histogram(salience, x = 'salience')

    plt_d.add_vline(x=np.quantile(salience.salience, q = .95), line_dash="solid", line_color="#5d5d5d")
    plt_d.add_vline(x=np.quantile(salience.salience, q = .99), line_dash="dash",  line_color="#2a2a2a")
    plt_d.add_vline(x=np.quantile(salience.salience, q = .999),line_dash="dot",   line_color="#000000")

    plt_d.write_html(save_dir+f"{plt_prefix}_dist.html")
    plt_d.write_image(save_dir+f"{plt_prefix}_dist.svg")

    # salience manhattan
    plt_m = px.scatter(salience, x = 'index', y = 'salience', color = 'chrom', hover_data=['pos', 'cxn'])

    plt_m.add_hline(y=np.quantile(salience.salience, q = .95), line_dash="solid", line_color="#5d5d5d")
    plt_m.add_hline(y=np.quantile(salience.salience, q = .99), line_dash="dash",  line_color="#2a2a2a")
    plt_m.add_hline(y=np.quantile(salience.salience, q = .999),line_dash="dot",   line_color="#000000")

    plt_m.write_html(save_dir+f"{plt_prefix}_manhattan.html")
    plt_m.write_image(save_dir+f"{plt_prefix}_manhattan.svg")

    return plt_m, plt_d



In [ ]:
#| export


def _collapse_and_add_gff_annotations(acgt_loci, # = acgt_loci, 
                            gene_nodes_gff, # = gene_nodes_gff, 
                            e, # = validation_inp_sals
                            ):
    salience = acgt_loci.copy()
    salience['salience'] = e.max(axis = 0) # max over observation axis
    salience = salience.reset_index()

    salience['chrom'] = salience['chrom'].astype(str)  
    salience['pos']   = salience['pos'].astype(int)  
    salience['cxn'] = ''
    for i in gene_nodes_gff.index:
        chrom, start, end, cxn_val = gene_nodes_gff.loc[i, ['chromosome', 'start', 'end', 'cxn']]
        salience.loc[(
            (salience.chrom == str(chrom)) &
            ((salience.pos  >= int(start)) & 
                (salience.pos  <= int(end)))
            ), 'cxn'] = cxn_val
    return salience



In [ ]:
#| export


# NOTE this could also be broken up into snpwise and genewise options
### Gene-wise Manhattan
def _collapse_salience_genewise(
        M_list, 
        acgt_loci, 
        sals,
        params_data
        ):
    # I can get the gene associations back like this. Not the same as a manhattan
    _ = pd.DataFrame([(
            e, 
            int(M_list[0].row_info[e]['start']),
            int(M_list[0].row_info[e]['stop'])
        ) for e in M_list[0].row_info.keys()], 
        columns=['cxn', 'start', 'stop']
        ).sort_values('start'
        ).reset_index(drop = True)

    _['start'] = (_['start'] / params_data['num_nucleotides']).astype(int) # M_list is in reference to the flattened data.
    _['stop']  = (_['stop']  / params_data['num_nucleotides']).astype(int)

    _['chrom'] = ''
    _['pos'] = 0
    _['max_sal'] = 0.0

    for i in _.index:
        start, stop = _.loc[i, ['start', 'stop']]
        _.loc[i, ['salience']] = float( sals[:, start:stop].max().item() )
        _.loc[i, ['chrom', 'pos']]  = acgt_loci.loc[round(np.mean([start, stop])), ['chrom','pos']]

    _['chrom'] = _['chrom'].astype(str) 
    _['pos'] = _['pos'].astype(str) 
    salience = _.reset_index()

    return salience



In [ ]:
#| export



# break this into two problems: 
    #   1. Collecting Gradients
    #   2. Organizing Gradients
    # 
    # Training a model with the same hyperparameters sometimes results in gradients that are max 0 and sometimes not. 
    # This seems to be a gradient attenuation problem. On one run I got max(abs(grads)) that look like so:
    # trn 0 -> 0 -> 0 -> 0 -> 0 -> 0 -> 0 -> 0.000 -> 0.001 -> 0.001
    # tst 0 -> 0 -> 0 -> 0 -> 0 -> 0 -> 0 -> 0.010 -> 0.031 -> 0.017
    # Observed with gmx data and params: 
    # "{'default_decay_rate': 0, 'default_drop_nodes_edge': 0.0, 'default_drop_nodes_inp': 0.0, 'default_drop_nodes_out': 0.0, 'default_out_nodes_edge': 2, 
    #   'default_out_nodes_inp': 1, 'default_out_nodes_out': 2, 'default_reps_nodes_edge': 2, 'default_reps_nodes_inp': 1, 'default_reps_nodes_out': 1}"
def collect_gradients(model, inp_dl):      
    if next(iter(inp_dl))[0].get_device() != -1:
        model = model.to('cuda')         
    "Returns a tuple of weight grads, bias grads"
    # Setup list of lists [layer, ..., layer] with batch in layer
    gradient_weight_holder = [[] for i in model.layer_list]
    gradient_bias_holder = [[] for i in model.layer_list]
    gradient_obs = []

    for i, (y_i, x_i) in enumerate(inp_dl):
        gradient_obs.append(len(y_i))
        # set to train mode, setup optimizer
        model = model.train()

        optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)
        loss_fn = nn.MSELoss()

        loss = loss_fn(model(x_i), y_i)
        optimizer.zero_grad()
        loss.backward()

        # now go through each of the layers and pull the gradient. 
        # Convert to numpy and save
        for level in range(len(model.layer_list)):
            gradient_weight_holder[level].append( model.layer_list[level].weights.grad.detach().cpu().numpy() )
            gradient_bias_holder[level].append( model.layer_list[level].bias.grad.detach().cpu().numpy() )

    gradient_obs = np.array(gradient_obs)
    # convert to percent of training set
    gradient_obs = gradient_obs/gradient_obs.sum()
    gradient_obs = gradient_obs[:, None]

    def _scale_gradients(gradient_list, gradient_obs): # gradient list should be gradient_holder[-1]
        _ = np.concatenate(gradient_list, 0).reshape(gradient_obs.shape[0], -1)
        _ = _ * gradient_obs # Weight by the number of obs that went into the gradient
        _ = _.sum(axis = 0)  # Get average gradient
        return _

    # turn batches of gradients in gradient_holder into average (accumulated) gradient
    gradient_weight_holder = [_scale_gradients(gradient_list = e, gradient_obs = gradient_obs) for e in gradient_weight_holder]
    gradient_bias_holder = [_scale_gradients(gradient_list = e, gradient_obs = gradient_obs) for e in gradient_bias_holder]
    
    return gradient_weight_holder, gradient_bias_holder


In [ ]:
#| export



def _collect_rho(model, M_list, inp_dl, y_names):
    if next(iter(inp_dl))[0].get_device() != -1:
        model = model.to('cuda')

    # Convert M_list into a usable lookup
    out = []
    for i in tqdm.tqdm(range(len(M_list)), ascii = True, desc = 'Building lookup table'):
        # break down a `structured_layer_info` class into a df
        slinfo = M_list[i]
        _ = [(k, int(slinfo.row_info[k]['start']), int(slinfo.row_info[k]['stop'])) for k in slinfo.row_info.keys()]
        _ = pd.DataFrame(_, columns=['node', 'start', 'stop'])
        _['layer'] = i
        out.append(_)

    out = pd.concat(out)

    out = out.reset_index(drop=True).reset_index().rename(columns={'index':'node_forward_idx'})

    # table with rows being (nodes x outputs per node) and cols being (obs) 
    output_tracker = pd.DataFrame(
        {'node_forward_idx' : sum(
            [[k for i in range(n)] 
                for k,n in zip(
                    out['node_forward_idx'].tolist(),
                    (out['stop'] - out['start']).tolist())
                    ], 
                    [])}
    )

    # Improved verison
    _ = []
    for node in out.node_forward_idx:
        mask = (out.node_forward_idx == node)

        start = out.loc[mask, 'start'].values[0].astype(int)
        stop  = out.loc[mask, 'stop' ].values[0].astype(int)

        _.append(
        pd.DataFrame({
            'node_forward_idx': out.loc[mask, 'node_forward_idx' ].values[0].astype(int), 
            'node':  out.loc[mask, 'node' ].values[0], 
            'start': out.loc[mask, 'start' ].values[0].astype(int), 
            'stop':  out.loc[mask, 'stop' ].values[0].astype(int), 
            'layer': out.loc[mask, 'layer' ].values[0].astype(int),                            
            'idx':   [i for i in range(start, stop)]
            })
        )
    _ = pd.concat(_, axis = 0).reset_index(drop = True)
    # make sure everything is sorted
    _ = _.sort_values(['layer', 'idx']).reset_index(drop = True)
    lookup = _.copy()


    inp_dl_G = inp_dl.dataset.G
    y_actual = [] # Track the actual y and the...
    cached_tensor_out = []
    cached_tensor_idx = []
    for i, (y,x) in tqdm.tqdm(enumerate(inp_dl), ascii = True, desc = 'Conducting forward pass'):
        # Find gene index in G (inp_dl_G) so we can store lookup values
        # i_idx = [torch.where((x[j, :] == inp_dl_G).all(dim=1))[0] for j in range(len(x))]
        # i_idx = torch.concatenate(i_idx).cpu().detach().numpy()
        # There's a potential bug here. It's possible for some data to be identical after filtering (seen in gmx)
        # In this case we return >1 value for where and break. The solution is to default to the minimum index OR 
        # to have this be a list and apply the values to all matching entries in inp_dl_G
        i_idx = [torch.where((x[j, :] == inp_dl_G).all(dim=1))[0] for j in range(len(x))]
        i_idx = [e.cpu().detach().tolist() for e in i_idx]
        # switching to using a list of lists (ideally each containing only one entry)
        if False not in [jj in cached_tensor_idx for j in i_idx for jj in j]:
            cached_tensor_idx = cached_tensor_idx+i_idx 
            y_actual          = np.concatenate([ y_actual, y.swapaxes(0,1).cpu().detach().numpy()], axis=1)

        else:
            # This is the .forward() method adapted to store all the interediate tensors
            with torch.no_grad():
                tensor_out = [x]
                for L in model.layer_list:
                    tensor_out.append(L(tensor_out[-1]))

            tensor_out = [e.cpu().detach().numpy() for e in tensor_out]
    
            _ = []
            for layer in sorted(list(set(lookup.layer))):
                _.append( tensor_out[layer][:, lookup.loc[(lookup.layer == layer), 'idx'].tolist()] )
            tmp = np.concatenate(_, axis=1)#.swapaxes(0,1)

            if type(y_actual) is list:
                cached_tensor_idx = i_idx
                y_actual = y.swapaxes(0,1).cpu().detach().numpy()
                cached_tensor_out = np.zeros((len(inp_dl_G), tmp.shape[1]))

            else:
                # cached_tensor_idx = np.concatenate([cached_tensor_idx, i_idx], axis=0) # 1d 
                cached_tensor_idx = cached_tensor_idx+i_idx 
                y_actual          = np.concatenate([ y_actual, y.swapaxes(0,1).cpu().detach().numpy()], axis=1)
                # This is doing way more operations than we need but it might be okay. 
                # Ideally I would check if the index has already been cached. If so we don't need to calculate it or save it. 
                # cached_tensor_out[i_idx, ] = tmp

            # Add in the recovered values (even if they map to multiple inputs)
            if 1 == max([len(e) for e in i_idx]):
                # if i_idx's sub lists contain only one value then we can collapse the list and use it like an array. 
                cached_tensor_out[sum(i_idx, []), ] = tmp
            else:
                # If not then we have to iterate through 
                for j in range(len(tmp)):
                    for jj in i_idx[j]:
                        cached_tensor_out[jj, ] = tmp[j]


    # Collapse to summary statistics
    rho_val = np.zeros((y_actual.shape[0], cached_tensor_out.shape[1]))
    rho_sig = np.zeros_like(rho_val)


    cached_tensor_idx = sum([[e[0]] for e in cached_tensor_idx], []) # if there are duplicate associations for a input arbitraily retain the 0th lookup.
    for y_i in tqdm.tqdm(range(rho_val.shape[0]), ascii = True, desc = 'Calculating spearman\'s rho for all y vars'):
        for n_i in tqdm.tqdm(range(rho_val.shape[1]), leave = False, ascii = True, desc = '... and intermediates'):
            if ((y_actual[y_i, :].std() == 0.0) | 
                (cached_tensor_out[cached_tensor_idx, n_i].std() == 0.0)): # expand out tensor to account for duplicate inputs
                # (node_out[n_i, :].std() == 0.0)):
                val, sig = np.nan, np.nan
            
            else:
                val, sig = scipy.stats.spearmanr( y_actual[y_i, :], cached_tensor_out[cached_tensor_idx, n_i] )
            
            rho_val[y_i, n_i] = val
            rho_sig[y_i, n_i] = sig

    output_tracker = pd.concat([
        output_tracker, 
        pd.DataFrame(rho_val.swapaxes(0,1), columns=y_names),
        pd.DataFrame(rho_sig.swapaxes(0,1), columns=[e+'_sig' for e in y_names])    
        ], axis=1)

    # Add lookup info (node names)
    output_tracker = lookup.loc[:, ['node_forward_idx', 'node']].drop_duplicates().merge(output_tracker)
    return output_tracker

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()